# Excipient microbiome classification analysis

This notebook contains the analysis supporting the MSc dissertation *Microbiome-Based Evaluation and Knowledge Mining of Pharmaceutical Excipients for Precision Formulation Design*. It covers data matching, molecular featurisation, model comparison, statistical validation, feature analysis and uncertainty ranking.


In [ ]:
import os
import re
import warnings
from pathlib import Path
from difflib import get_close_matches

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem

from mordred import Calculator, descriptors

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, recall_score, make_scorer
from scipy.stats import fisher_exact

import joblib

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_WARNINGS"] = "0"

SEED = 42
np.random.seed(SEED)

try:
    from adjustText import adjust_text
    USE_ADJUST_TEXT = True
except:
    USE_ADJUST_TEXT = False
    print("adjustText not installed. Labels may overlap.")

print("Environment ready.")
print("Seed:", SEED)

In [ ]:
repo_root = Path.cwd()
data_folder = repo_root / "data"
output_folder = repo_root / "outputs"
figure_folder = output_folder / "figures"
checkpoint_folder = output_folder / "checkpoints"

for folder in [data_folder, output_folder, figure_folder, checkpoint_folder]:
    folder.mkdir(parents=True, exist_ok=True)

search_dirs = [data_folder, repo_root]


def normalise_filename(name):
    return str(name).replace(" ", "").replace("_", "").lower()


def find_excel_file(priority_names, patterns, search_dirs):
    # Exact or near-exact search first.
    priority_keys = [normalise_filename(x) for x in priority_names]

    for folder in search_dirs:
        if not folder.exists():
            continue

        for path in folder.glob("*.xls*"):
            if normalise_filename(path.name) in priority_keys:
                return path

    # Pattern search second.
    candidates = []

    for folder in search_dirs:
        if not folder.exists():
            continue

        for pattern in patterns:
            candidates.extend(list(folder.glob(pattern)))

    candidates = [
        candidate for candidate in candidates
        if candidate.suffix.lower() in [".xlsx", ".xls"]
    ]

    if not candidates:
        raise FileNotFoundError(
            f"No Excel file found for {priority_names}. "
            f"Place the required file in {data_folder.resolve()}."
        )

    return sorted(candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]


phase1_path = find_excel_file(
    priority_names=[
        "Phase1_Data_Mining_Excipient_List_Original_Style_Updated.xlsx"
    ],
    patterns=[
        "Phase1_Data_Mining_Excipient_List_Original_Style_Updated*.xlsx",
        "Phase1_Data_Mining_Excipient*.xlsx"
    ],
    search_dirs=search_dirs
)

label_path = find_excel_file(
    priority_names=[
        "Book 16.xlsx",
        "Book16.xlsx",
        "Book 16(1).xlsx",
        "Book16(1).xlsx",
        "Book 16 (1).xlsx",
        "Book16 (1).xlsx"
    ],
    patterns=[
        "Book*16*.xlsx",
        "Book16*.xlsx"
    ],
    search_dirs=search_dirs
)

phase1_excel = pd.ExcelFile(phase1_path)
phase1_sheet = (
    "Excipients" if "Excipients" in phase1_excel.sheet_names
    else phase1_excel.sheet_names[0]
)

label_excel = pd.ExcelFile(label_path)
label_sheet = label_excel.sheet_names[0]

phase1 = pd.read_excel(phase1_path, sheet_name=phase1_sheet)
labels = pd.read_excel(label_path, sheet_name=label_sheet)

print("Phase 1 file:", phase1_path)
print("Phase 1 sheet:", phase1_sheet)
print("Phase 1 shape:", phase1.shape)

print("\nLabel file:", label_path)
print("Label sheet:", label_sheet)
print("Label shape:", labels.shape)

print("\nPhase 1 columns:")
print(phase1.columns.tolist())

print("\nLabel columns:")
print(labels.columns.tolist())

print("\nOutput folder:")
print(output_folder)


In [ ]:
def find_column(df, possible_names):
    col_map = {str(c).strip().lower(): c for c in df.columns}
    
    for name in possible_names:
        key = name.strip().lower()
        if key in col_map:
            return col_map[key]
    
    for c in df.columns:
        c_lower = str(c).strip().lower()
        for name in possible_names:
            if name.strip().lower() in c_lower:
                return c
    
    raise ValueError(f"Could not find column from options: {possible_names}")


phase1_name_col = find_column(
    phase1,
    ["Excipient Name", "Excipient", "Ingredient Name", "Name"]
)

phase1_smiles_col = find_column(
    phase1,
    ["SMILES", "Canonical SMILES", "Structure"]
)

label_name_col = find_column(
    labels,
    ["Excipient Name", "Excipient", "Ingredient Name", "Name"]
)

label_col = find_column(
    labels,
    ["Label", "Final_Label", "Final Label", "Microbiome Label", "Effect Label", "Class"]
)


def clean_name(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).upper().strip()
    x = x.replace("\xa0", " ")
    x = x.replace("–", "-").replace("—", "-")
    x = x.replace("’", "'").replace("‘", "'")
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^A-Z0-9\s\-\.,\(\)/&]", "", x)
    
    return x.strip()


def clean_label(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).upper().strip()
    x = x.replace("-", " ")
    x = x.replace("_", " ")
    x = re.sub(r"\s+", " ", x)
    
    if "PROMOTE" in x:
        return "Promote"
    if "NEUTRAL" in x:
        return "Neutral"
    if "PREVENT" in x or "INHIBIT" in x:
        return "Prevent"
    
    return np.nan


def canonicalise_smiles(smiles):
    if pd.isna(smiles):
        return np.nan
    
    smiles = str(smiles).strip()
    
    invalid_values = {
        "",
        "NAN",
        "NONE",
        "NA",
        "N/A",
        "UNKNOWN",
        "MISSING",
        "NULL",
        "-",
        "NOT APPLICABLE",
        "NOT APPLICABLE - MIXTURE/POLYMER/PROPRIETARY GRADE"
    }
    
    if smiles.upper() in invalid_values:
        return np.nan
    
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return np.nan
    
    return Chem.MolToSmiles(mol, canonical=True)


print("Detected columns:")
print("Phase 1 name column:", phase1_name_col)
print("Phase 1 SMILES column:", phase1_smiles_col)
print("Label name column:", label_name_col)
print("Label column:", label_col)

In [ ]:
phase1 = phase1.copy()
labels = labels.copy()

phase1["name_key"] = phase1[phase1_name_col].apply(clean_name)
labels["name_key"] = labels[label_name_col].apply(clean_name)
labels["Label_clean"] = labels[label_col].apply(clean_label)

alias_map = {
    "CALCIUM CHLORIDE": "CALCIUM CHLORIDE DIHYDRATE",
    "DEXTROSE UNSPECIFIED FORM": "ANHYDROUS DEXTROSE",
    "DEXTROSE, UNSPECIFIED FORM": "ANHYDROUS DEXTROSE",
    "LACTOSE UNSPECIFIED FORM": "LACTOSE MONOHYDRATE",
    "LACTOSE, UNSPECIFIED FORM": "LACTOSE MONOHYDRATE",
    "LACTOSE ANHYDROUS": "ANHYDROUS LACTOSE",
    "MICROCRYSTALLINE CELLULOSE": "CELLULOSE, MICROCRYSTALLINE",
    "FD&C YELLOW 5": "FD&C YELLOW NO. 5",
    "FD&C RED 40": "FD&C RED NO. 40",
    "D&C ORANGE 4": "D&C ORANGE NO. 4",
    "ACACIA SYRUP": "ACACIA",
    "POLYDEXTROSE K": "POLYDEXTROSE"
}

labels["name_key"] = labels["name_key"].replace(alias_map)

phase1["canonical_smiles"] = phase1[phase1_smiles_col].apply(canonicalise_smiles)

print("Cleaned label distribution:")
print(labels["Label_clean"].value_counts(dropna=False))

print("\nPhase 1 valid SMILES:")
print(phase1["canonical_smiles"].notna().sum())

labels_usable = labels.dropna(subset=["name_key", "Label_clean"]).copy()

label_summary = (
    labels_usable
    .groupby("name_key")
    .agg(
        Label_File_Name=(label_name_col, "first"),
        Labels=("Label_clean", lambda x: sorted(set(x))),
        N_label_records=("Label_clean", "size")
    )
    .reset_index()
)

label_summary["Label_growth"] = label_summary["Labels"].apply(
    lambda x: x[0] if len(x) == 1 else "CONFLICT"
)

label_conflicts = label_summary[label_summary["Label_growth"] == "CONFLICT"].copy()
label_final = label_summary[label_summary["Label_growth"] != "CONFLICT"].copy()

matched_all = label_final.merge(
    phase1,
    on="name_key",
    how="left"
)

matched_all["canonical_smiles"] = matched_all[phase1_smiles_col].apply(canonicalise_smiles)

matched = matched_all[matched_all[phase1_smiles_col].notna()].copy()
unmatched = matched_all[matched_all[phase1_smiles_col].isna()].copy()

labelled_valid = matched[matched["canonical_smiles"].notna()].copy()
labelled_invalid_smiles = matched[matched["canonical_smiles"].isna()].copy()

# Keep one training record per labelled excipient name.
# This avoids accidental inflation if Phase 1 contains multiple grade records for one labelled excipient.
labelled_valid = (
    labelled_valid
    .sort_values(["name_key", "canonical_smiles"])
    .drop_duplicates(subset=["name_key"], keep="first")
    .reset_index(drop=True)
)

phase1_name_list = sorted(phase1["name_key"].dropna().unique().tolist())

def closest_phase1_name(name):
    matches = get_close_matches(name, phase1_name_list, n=1, cutoff=0.60)
    return matches[0] if matches else np.nan

if unmatched.shape[0] > 0:
    unmatched["Closest_Phase1_Name"] = unmatched["name_key"].apply(closest_phase1_name)

print("Input label rows:", labels.shape[0])
print("Usable labels after cleaning:", label_final.shape[0])
print("Conflicting labels excluded:", label_conflicts.shape[0])
print("Matched rows before deduplication:", matched.shape[0])
print("Unmatched labels:", unmatched.shape[0])
print("Valid labelled records after one-record-per-label rule:", labelled_valid.shape[0])
print("Invalid labelled SMILES:", labelled_invalid_smiles.shape[0])

print("\nFinal labelled class distribution:")
print(labelled_valid["Label_growth"].value_counts())

display(unmatched)
display(labelled_valid[["Label_File_Name", "Label_growth", "canonical_smiles"]])

In [ ]:
labelled_for_ml = labelled_valid.copy()
labelled_for_ml["Display_Name"] = labelled_for_ml["Label_File_Name"]

labelled_smiles = set(labelled_for_ml["canonical_smiles"].dropna())
labelled_names = set(labelled_for_ml["name_key"].dropna())

# Conservative removal:
# remove labelled names and also remove structures already represented in the labelled set.
unlabelled_pool = phase1[
    (~phase1["name_key"].isin(labelled_names)) &
    (~phase1["canonical_smiles"].isin(labelled_smiles))
].copy()

unlabelled_valid = unlabelled_pool[
    unlabelled_pool["canonical_smiles"].notna()
].copy()

print("Final labelled samples for ML:", labelled_for_ml.shape[0])
print(labelled_for_ml["Label_growth"].value_counts())

print("\nUnlabelled Phase 1 records with valid SMILES:")
print(unlabelled_valid.shape[0])

print("\nUnique canonical SMILES in prediction pool:")
print(unlabelled_valid["canonical_smiles"].nunique())

display(labelled_for_ml[["Display_Name", "Label_growth", "canonical_smiles"]])

In [ ]:
all_smiles = pd.concat(
    [
        labelled_for_ml["canonical_smiles"],
        unlabelled_valid["canonical_smiles"]
    ],
    ignore_index=True
).dropna().drop_duplicates().reset_index(drop=True)

print("Unique SMILES to featurise:", len(all_smiles))

mols = [Chem.MolFromSmiles(s) for s in all_smiles]

calc = Calculator(descriptors, ignore_3D=True)

mordred_df = calc.pandas(mols, nproc=1)
mordred_df.index = all_smiles
mordred_df.columns = mordred_df.columns.astype(str)

mordred_df = mordred_df.apply(pd.to_numeric, errors="coerce")

print("Mordred descriptor matrix:", mordred_df.shape)
display(mordred_df.head())

In [ ]:
def calculate_morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return np.full(n_bits, np.nan)
    
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )
    
    arr = np.zeros((n_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    
    return arr


morgan_array = np.vstack([
    calculate_morgan_fp(s)
    for s in all_smiles
])

morgan_df = pd.DataFrame(
    morgan_array,
    index=all_smiles,
    columns=[f"Morgan_{i}" for i in range(2048)]
)

print("Morgan fingerprint matrix:", morgan_df.shape)
display(morgan_df.head())

In [ ]:
molecular_features = pd.concat(
    [
        mordred_df,
        morgan_df
    ],
    axis=1
)

molecular_features.columns = molecular_features.columns.astype(str)

categorical_cols = [
    "Functional role",
    "Chemical Family",
    "Oral Dosage Form",
    "Polymer Class",
    "Polymer Flag",
    "Viscosity",
    "Aqueous Solubility",
    "Solubility Class"
]

numeric_cols = [
    "Molecular Weight"
]

categorical_cols = [c for c in categorical_cols if c in phase1.columns]
numeric_cols = [c for c in numeric_cols if c in phase1.columns]

print("Categorical metadata used:")
print(categorical_cols)

print("\nNumeric metadata used:")
print(numeric_cols)

labelled_base = labelled_for_ml.copy()
labelled_base["_dataset"] = "labelled"

pool_base = unlabelled_valid.copy()
pool_base["_dataset"] = "pool"

all_rows = pd.concat(
    [
        labelled_base,
        pool_base
    ],
    ignore_index=True,
    sort=False
)

mol_part = all_rows[["canonical_smiles"]].merge(
    molecular_features,
    left_on="canonical_smiles",
    right_index=True,
    how="left"
).drop(columns=["canonical_smiles"])

cat_part = pd.get_dummies(
    all_rows[categorical_cols].astype("string").fillna("Missing"),
    prefix=categorical_cols,
    dtype=int
)

num_part = pd.DataFrame(index=all_rows.index)

for col in numeric_cols:
    num_part[col] = pd.to_numeric(all_rows[col], errors="coerce")

X_all = pd.concat(
    [
        mol_part.reset_index(drop=True),
        cat_part.reset_index(drop=True),
        num_part.reset_index(drop=True)
    ],
    axis=1
)

X_all.columns = X_all.columns.astype(str)
X_all = X_all.loc[:, ~X_all.columns.duplicated()]
X_all = X_all.replace([np.inf, -np.inf], np.nan)

feature_cols = X_all.columns.tolist()

labelled_desc = all_rows[all_rows["_dataset"] == "labelled"].copy().reset_index(drop=True)
unlabelled_desc = all_rows[all_rows["_dataset"] == "pool"].copy().reset_index(drop=True)

X_raw = X_all.loc[all_rows["_dataset"] == "labelled"].copy().reset_index(drop=True)
X_pool_raw = X_all.loc[all_rows["_dataset"] == "pool"].copy().reset_index(drop=True)

y = labelled_desc["Label_growth"].copy().reset_index(drop=True)

print("Hybrid training matrix:", X_raw.shape)
print("Hybrid unlabelled pool matrix:", X_pool_raw.shape)
print("Duplicated columns:", X_raw.columns.duplicated().sum())

print("\nTraining labels:")
print(y.value_counts())

In [ ]:
class MissingnessFilter(BaseEstimator, TransformerMixin):
    def __init__(self, max_missingness=0.40):
        self.max_missingness = max_missingness
    
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.keep_columns_ = X.columns[
            X.isna().mean() <= self.max_missingness
        ].tolist()
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X)
        return X[self.keep_columns_]
    
    def get_feature_names_out(self):
        return np.array(self.keep_columns_)


class SafeSelectKBest(BaseEstimator, TransformerMixin):
    def __init__(self, k=100):
        self.k = k
    
    def fit(self, X, y):
        if self.k == "all":
            self.selector_ = "passthrough"
            self.mask_ = np.ones(X.shape[1], dtype=bool)
            return self
        
        safe_k = min(int(self.k), X.shape[1])
        self.selector_ = SelectKBest(f_classif, k=safe_k)
        self.selector_.fit(X, y)
        self.mask_ = self.selector_.get_support()
        return self
    
    def transform(self, X):
        if self.selector_ == "passthrough":
            return X
        return self.selector_.transform(X)
    
    def get_support(self):
        return self.mask_


MAX_MISSINGNESS = 0.40

print("Preprocessing tools ready.")

In [ ]:
visual_pipe = Pipeline([
    ("missingness", MissingnessFilter(MAX_MISSINGNESS)),
    ("imputer", SimpleImputer(strategy="median")),
    ("variance", VarianceThreshold(0.0)),
    ("scaler", StandardScaler())
])

X_visual_labelled = visual_pipe.fit_transform(X_raw)

kept_after_missingness = np.array(visual_pipe.named_steps["missingness"].keep_columns_)
variance_mask = visual_pipe.named_steps["variance"].get_support()
visual_feature_names = kept_after_missingness[variance_mask]

pca = PCA(n_components=2, random_state=SEED)
pca_xy = pca.fit_transform(X_visual_labelled)

pca_df = pd.DataFrame({
    "Excipient": labelled_desc["Display_Name"].values,
    "Label_growth": y.values,
    "PC1": pca_xy[:, 0],
    "PC2": pca_xy[:, 1]
})

pca_var1 = pca.explained_variance_ratio_[0] * 100
pca_var2 = pca.explained_variance_ratio_[1] * 100

color_map = {
    "Promote": "#2E8B57",
    "Neutral": "#376A9F",
    "Prevent": "#C44E52"
}

fig, ax = plt.subplots(figsize=(9, 7))

texts = []

for label in ["Promote", "Neutral", "Prevent"]:
    sub = pca_df[pca_df["Label_growth"] == label]
    
    ax.scatter(
        sub["PC1"],
        sub["PC2"],
        s=90,
        color=color_map[label],
        label=label.upper(),
        alpha=0.95,
        edgecolor="black",
        linewidth=0.5
    )
    
    for _, row in sub.iterrows():
        texts.append(
            ax.text(
                row["PC1"],
                row["PC2"],
                str(row["Excipient"]).upper(),
                fontsize=7,
                alpha=0.9
            )
        )

if USE_ADJUST_TEXT:
    adjust_text(
        texts,
        arrowprops=dict(
            arrowstyle="-",
            color="gray",
            lw=0.5,
            alpha=0.6
        )
    )

ax.set_title("PCA labelled excipients only", fontsize=14, fontweight="bold")
ax.set_xlabel(f"PC1 ({pca_var1:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca_var2:.1f}% variance)")
ax.legend(frameon=False)
ax.grid(alpha=0.25)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

pca_png = figure_folder / "PCA_labelled_excipients_only.png"
pca_pdf = figure_folder / "PCA_labelled_excipients_only.pdf"

plt.savefig(pca_png, dpi=300, bbox_inches="tight")
plt.savefig(pca_pdf, bbox_inches="tight")

print("Saved PCA PNG:", pca_png)
print("Saved PCA PDF:", pca_pdf)

plt.show()

In [ ]:
pca_loadings = pd.DataFrame({
    "Feature": visual_feature_names,
    "PC1_loading": pca.components_[0],
    "PC2_loading": pca.components_[1]
})

pca_loadings["PC1_abs_loading"] = pca_loadings["PC1_loading"].abs()
pca_loadings["PC2_abs_loading"] = pca_loadings["PC2_loading"].abs()
pca_loadings["Combined_abs_loading"] = (
    pca_loadings["PC1_abs_loading"] + pca_loadings["PC2_abs_loading"]
)

pca_loadings = pca_loadings.sort_values(
    "Combined_abs_loading",
    ascending=False
).reset_index(drop=True)

display(pca_df)
display(pca_loadings.head(30))

In [ ]:
def promote_recall(y_true, y_pred):
    return recall_score(
        y_true,
        y_pred,
        labels=["Promote"],
        average="macro",
        zero_division=0
    )


promote_scorer = make_scorer(promote_recall)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced",
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1
    ),
    
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=250,
        class_weight="balanced",
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1
    ),
    
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED
    ),
    
    "SVM RBF C1": SVC(
        C=1,
        kernel="rbf",
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=SEED
    ),
    
    "SVM RBF C10": SVC(
        C=10,
        kernel="rbf",
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=SEED
    )
}


def make_model(clf, k):
    return Pipeline([
        ("missingness", MissingnessFilter(MAX_MISSINGNESS)),
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(0.0)),
        ("scaler", StandardScaler()),
        ("select", SafeSelectKBest(k)),
        ("model", clf)
    ])


min_class_count = y.value_counts().min()
n_splits = min(3, int(min_class_count))

if n_splits < 2:
    raise ValueError("Not enough samples per class for cross-validation.")

cv_main = RepeatedStratifiedKFold(
    n_splits=n_splits,
    n_repeats=5,
    random_state=SEED
)

scoring_dict = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "promote_recall": promote_scorer
}

feature_options = [10, 20, 50, 100, "all"]

results = []

for model_name, clf in models.items():
    for k in feature_options:
        print("Running:", model_name, "| selected features =", k)
        
        pipe = make_model(clone(clf), k)
        
        scores = cross_validate(
            pipe,
            X_raw,
            y,
            cv=cv_main,
            scoring=scoring_dict,
            n_jobs=1
        )
        
        results.append({
            "Model": model_name,
            "Selected_features": k,
            "Accuracy_mean": scores["test_accuracy"].mean(),
            "Accuracy_sd": scores["test_accuracy"].std(),
            "Balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
            "Balanced_accuracy_sd": scores["test_balanced_accuracy"].std(),
            "Macro_F1_mean": scores["test_macro_f1"].mean(),
            "Macro_F1_sd": scores["test_macro_f1"].std(),
            "Promote_recall_mean": scores["test_promote_recall"].mean(),
            "Promote_recall_sd": scores["test_promote_recall"].std()
        })

model_comparison = pd.DataFrame(results)

model_comparison["Selection_score"] = (
    0.40 * model_comparison["Balanced_accuracy_mean"] +
    0.35 * model_comparison["Macro_F1_mean"] +
    0.25 * model_comparison["Promote_recall_mean"]
)

model_comparison = model_comparison.sort_values(
    by=["Selection_score", "Balanced_accuracy_mean", "Macro_F1_mean"],
    ascending=False
).reset_index(drop=True)

best = model_comparison.iloc[0]

best_model_name = best["Model"]
best_k = best["Selected_features"]
best_accuracy = best["Accuracy_mean"]
best_balanced_accuracy = best["Balanced_accuracy_mean"]
best_macro_f1 = best["Macro_F1_mean"]
best_promote_recall = best["Promote_recall_mean"]
best_selection_score = best["Selection_score"]

observed_max_balanced_accuracy = model_comparison["Balanced_accuracy_mean"].max()

display(model_comparison)

print("Best model by composite score:", best_model_name)
print("Best selected features:", best_k)
print("Accuracy:", round(best_accuracy, 3))
print("Balanced accuracy:", round(best_balanced_accuracy, 3))
print("Macro F1:", round(best_macro_f1, 3))
print("Promote recall:", round(best_promote_recall, 3))
print("Selection score:", round(best_selection_score, 3))
print("Maximum balanced accuracy across 25 configurations:", round(observed_max_balanced_accuracy, 3))

In [ ]:
selected_pipe = make_model(
    clone(models[best_model_name]),
    best_k
)

cv_simple = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

y_pred = cross_val_predict(
    selected_pipe,
    X_raw,
    y,
    cv=cv_simple
)

cv_prediction_df = labelled_desc.copy()
cv_prediction_df["True_Label"] = y.values
cv_prediction_df["CV_Predicted_Label"] = y_pred
cv_prediction_df["Correct"] = (
    cv_prediction_df["True_Label"] == cv_prediction_df["CV_Predicted_Label"]
)

labels_order = ["Neutral", "Prevent", "Promote"]
labels_order = [label for label in labels_order if label in sorted(y.unique())]

cm = confusion_matrix(
    y,
    y_pred,
    labels=labels_order
)

confusion_matrix_df = pd.DataFrame(
    cm,
    index=labels_order,
    columns=labels_order
)

classification_report_df = pd.DataFrame(
    classification_report(
        y,
        y_pred,
        output_dict=True,
        zero_division=0
    )
).transpose()


def wilson_score_interval(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    
    p = k / n
    denominator = 1 + (z ** 2) / n
    centre = (p + (z ** 2) / (2 * n)) / denominator
    margin = (
        z
        * np.sqrt((p * (1 - p) / n) + ((z ** 2) / (4 * n ** 2)))
        / denominator
    )
    
    lower = max(0, centre - margin)
    upper = min(1, centre + margin)
    
    return lower, upper


overall_correct = int((y.values == y_pred).sum())
overall_total = int(len(y))
overall_accuracy = overall_correct / overall_total
overall_ci = wilson_score_interval(overall_correct, overall_total)

ci_rows = []

for cls in labels_order:
    true_positive = int(((y.values == cls) & (y_pred == cls)).sum())
    actual_total = int((y.values == cls).sum())
    predicted_total = int((y_pred == cls).sum())
    
    recall = true_positive / actual_total if actual_total > 0 else np.nan
    precision = true_positive / predicted_total if predicted_total > 0 else np.nan
    
    recall_lower, recall_upper = wilson_score_interval(true_positive, actual_total)
    precision_lower, precision_upper = wilson_score_interval(true_positive, predicted_total)
    
    ci_rows.append({
        "Class": cls,
        "Support": actual_total,
        "Predicted_total": predicted_total,
        "Recall": recall,
        "Recall_CI_lower": recall_lower,
        "Recall_CI_upper": recall_upper,
        "Precision": precision,
        "Precision_CI_lower": precision_lower,
        "Precision_CI_upper": precision_upper
    })

per_class_ci = pd.DataFrame(ci_rows)

display(classification_report_df)
display(confusion_matrix_df)
display(per_class_ci)

print("Out-of-fold accuracy:", overall_accuracy)
print("Out-of-fold Wilson 95% CI:", overall_ci)

In [ ]:
baseline_models = {
    "Dummy most frequent": DummyClassifier(strategy="most_frequent"),
    "Dummy stratified": DummyClassifier(strategy="stratified", random_state=SEED),
    "Dummy uniform": DummyClassifier(strategy="uniform", random_state=SEED)
}

baseline_rows = []

selected_model = make_model(
    clone(models[best_model_name]),
    best_k
)

selected_scores = cross_validate(
    selected_model,
    X_raw,
    y,
    cv=cv_main,
    scoring=scoring_dict,
    n_jobs=1
)

baseline_rows.append({
    "Model": "Selected model",
    "Accuracy": selected_scores["test_accuracy"].mean(),
    "Balanced_accuracy": selected_scores["test_balanced_accuracy"].mean(),
    "Macro_F1": selected_scores["test_macro_f1"].mean(),
    "Promote_recall": selected_scores["test_promote_recall"].mean()
})

for name, dummy in baseline_models.items():
    scores = cross_validate(
        dummy,
        X_raw,
        y,
        cv=cv_main,
        scoring=scoring_dict,
        n_jobs=1
    )
    
    baseline_rows.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Balanced_accuracy": scores["test_balanced_accuracy"].mean(),
        "Macro_F1": scores["test_macro_f1"].mean(),
        "Promote_recall": scores["test_promote_recall"].mean()
    })

baseline_comparison = pd.DataFrame(baseline_rows)

display(baseline_comparison)

In [ ]:
N_MAXSTAT_PERMUTATIONS = 1000
SAVE_EVERY = 10

maxstat_checkpoint = checkpoint_folder / "max_statistic_permutation_checkpoint.csv"

def evaluate_25_configurations(y_input):
    rows = []
    
    for model_name, clf in models.items():
        for k in feature_options:
            pipe = make_model(clone(clf), k)
            
            scores = cross_validate(
                pipe,
                X_raw,
                y_input,
                cv=cv_main,
                scoring=scoring_dict,
                n_jobs=1
            )
            
            ba = scores["test_balanced_accuracy"].mean()
            mf1 = scores["test_macro_f1"].mean()
            pr = scores["test_promote_recall"].mean()
            acc = scores["test_accuracy"].mean()
            
            selection_score = (
                0.40 * ba +
                0.35 * mf1 +
                0.25 * pr
            )
            
            rows.append({
                "Model": model_name,
                "Selected_features": k,
                "Accuracy": acc,
                "Balanced_accuracy": ba,
                "Macro_F1": mf1,
                "Promote_recall": pr,
                "Selection_score": selection_score
            })
    
    return pd.DataFrame(rows)


if maxstat_checkpoint.exists():
    max_permutation_results = pd.read_csv(maxstat_checkpoint)
    completed = int(max_permutation_results["Permutation"].max())
    print("Resuming from existing checkpoint.")
    print("Completed permutations:", completed)
else:
    max_permutation_results = pd.DataFrame()
    completed = 0
    print("Starting new max-statistic permutation test.")

rng = np.random.default_rng(SEED + 999)

new_rows = []

for perm_id in range(completed + 1, N_MAXSTAT_PERMUTATIONS + 1):
    print(f"Max-statistic permutation {perm_id}/{N_MAXSTAT_PERMUTATIONS}")
    
    y_permuted = pd.Series(
        rng.permutation(y.values),
        index=y.index,
        name="Permuted_Label"
    )
    
    perm_config_df = evaluate_25_configurations(y_permuted)
    
    # 1. Selection-adjusted statistic:
    # repeat the same composite-score selection process under permuted labels.
    selected_by_score = perm_config_df.sort_values(
        by=["Selection_score", "Balanced_accuracy", "Macro_F1"],
        ascending=False
    ).iloc[0]
    
    # 2. Conservative max-balanced-accuracy statistic:
    # maximum BA across all 25 configurations under permuted labels.
    max_ba_row = perm_config_df.sort_values(
        by=["Balanced_accuracy", "Macro_F1"],
        ascending=False
    ).iloc[0]
    
    new_rows.append({
        "Permutation": perm_id,
        "Selected_by_score_Model": selected_by_score["Model"],
        "Selected_by_score_features": selected_by_score["Selected_features"],
        "Selected_by_score_Balanced_accuracy": selected_by_score["Balanced_accuracy"],
        "Selected_by_score_Macro_F1": selected_by_score["Macro_F1"],
        "Selected_by_score_Promote_recall": selected_by_score["Promote_recall"],
        "Selected_by_score_Selection_score": selected_by_score["Selection_score"],
        "Max_BA_Model": max_ba_row["Model"],
        "Max_BA_Selected_features": max_ba_row["Selected_features"],
        "Max_Balanced_accuracy": max_ba_row["Balanced_accuracy"],
        "Max_BA_Macro_F1": max_ba_row["Macro_F1"],
        "Max_BA_Promote_recall": max_ba_row["Promote_recall"],
        "Max_BA_Selection_score": max_ba_row["Selection_score"]
    })
    
    if perm_id % SAVE_EVERY == 0:
        temp_df = pd.concat(
            [
                max_permutation_results,
                pd.DataFrame(new_rows)
            ],
            ignore_index=True
        )
        temp_df.to_csv(maxstat_checkpoint, index=False)
        print("Checkpoint saved:", maxstat_checkpoint)

if len(new_rows) > 0:
    max_permutation_results = pd.concat(
        [
            max_permutation_results,
            pd.DataFrame(new_rows)
        ],
        ignore_index=True
    )

max_permutation_results = max_permutation_results.drop_duplicates(
    subset=["Permutation"],
    keep="last"
).sort_values("Permutation").reset_index(drop=True)

max_permutation_results.to_csv(maxstat_checkpoint, index=False)

# Selection-adjusted p-value:
# compares observed selected model BA with the BA obtained after repeating model selection under each permutation.
selection_adjusted_p_value = (
    1 + (
        max_permutation_results["Selected_by_score_Balanced_accuracy"] >= best_balanced_accuracy
    ).sum()
) / (max_permutation_results.shape[0] + 1)

# Conservative max-BA p-value:
# compares observed max BA across 25 configurations with max BA across 25 configurations under each permutation.
max_ba_p_value = (
    1 + (
        max_permutation_results["Max_Balanced_accuracy"] >= observed_max_balanced_accuracy
    ).sum()
) / (max_permutation_results.shape[0] + 1)

max_stat_summary = pd.DataFrame({
    "Metric": [
        "Observed selected model",
        "Observed selected features",
        "Observed selected balanced accuracy",
        "Observed max balanced accuracy across 25 configurations",
        "Mean selection-adjusted permuted balanced accuracy",
        "Mean max permuted balanced accuracy",
        "Selection-adjusted max-statistic p-value",
        "Conservative max-BA p-value",
        "Number of permutations completed"
    ],
    "Value": [
        best_model_name,
        best_k,
        best_balanced_accuracy,
        observed_max_balanced_accuracy,
        max_permutation_results["Selected_by_score_Balanced_accuracy"].mean(),
        max_permutation_results["Max_Balanced_accuracy"].mean(),
        selection_adjusted_p_value,
        max_ba_p_value,
        max_permutation_results.shape[0]
    ]
})

display(max_stat_summary)
display(max_permutation_results.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    max_permutation_results["Selected_by_score_Balanced_accuracy"],
    bins=20,
    alpha=0.75,
    label="Selection-adjusted null"
)

ax.axvline(
    best_balanced_accuracy,
    linestyle="--",
    linewidth=2,
    label=f"Observed selected model = {best_balanced_accuracy:.3f}"
)

ax.axvline(
    1/3,
    linestyle=":",
    linewidth=2,
    color="gray",
    label="Chance = 0.333"
)

ax.set_title("Max-statistic permutation test", fontweight="bold")
ax.set_xlabel("Balanced accuracy after repeating model selection\nunder permuted labels")
ax.set_ylabel("Frequency")
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

maxstat_png = figure_folder / "max_statistic_permutation_test.png"
maxstat_pdf = figure_folder / "max_statistic_permutation_test.pdf"

plt.savefig(maxstat_png, dpi=300, bbox_inches="tight")
plt.savefig(maxstat_pdf, bbox_inches="tight")

print("Saved:", maxstat_png)
print("Saved:", maxstat_pdf)

plt.show()

In [ ]:
morgan_cols = [
    col for col in X_raw.columns
    if str(col).startswith("Morgan_")
]

metadata_cols = []

for col in categorical_cols:
    metadata_cols += [
        feature for feature in X_raw.columns
        if str(feature).startswith(col + "_")
    ]

for col in numeric_cols:
    if col in X_raw.columns:
        metadata_cols.append(col)

metadata_cols = list(dict.fromkeys(metadata_cols))

mordred_cols = [
    col for col in X_raw.columns
    if col not in set(morgan_cols + metadata_cols)
]

print("Mordred features:", len(mordred_cols))
print("Morgan features:", len(morgan_cols))
print("Metadata features:", len(metadata_cols))

feature_sets = {
    "Mordred + Morgan": mordred_cols + morgan_cols,
    "Mordred only": mordred_cols,
    "Full hybrid": mordred_cols + morgan_cols + metadata_cols,
    "Mordred + metadata": mordred_cols + metadata_cols,
    "Morgan only": morgan_cols,
    "Morgan + metadata": morgan_cols + metadata_cols,
    "Metadata only": metadata_cols
}

ablation_rows = []

for feature_set_name, cols in feature_sets.items():
    cols = list(dict.fromkeys(cols))
    
    print("Running:", feature_set_name, "| features:", len(cols))
    
    model = make_model(
        clone(models["Random Forest"]),
        50
    )
    
    scores = cross_validate(
        model,
        X_raw[cols],
        y,
        cv=cv_main,
        scoring=scoring_dict,
        n_jobs=1
    )
    
    ablation_rows.append({
        "Feature_set": feature_set_name,
        "Number_of_features": len(cols),
        "Accuracy_mean": scores["test_accuracy"].mean(),
        "Accuracy_sd": scores["test_accuracy"].std(),
        "Balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "Balanced_accuracy_sd": scores["test_balanced_accuracy"].std(),
        "Macro_F1_mean": scores["test_macro_f1"].mean(),
        "Macro_F1_sd": scores["test_macro_f1"].std(),
        "Promote_recall_mean": scores["test_promote_recall"].mean(),
        "Promote_recall_sd": scores["test_promote_recall"].std()
    })

ablation_results = pd.DataFrame(ablation_rows)

full_ba = ablation_results.loc[
    ablation_results["Feature_set"] == "Full hybrid",
    "Balanced_accuracy_mean"
].iloc[0]

full_f1 = ablation_results.loc[
    ablation_results["Feature_set"] == "Full hybrid",
    "Macro_F1_mean"
].iloc[0]

ablation_results["Balanced_accuracy_drop_vs_full"] = (
    full_ba - ablation_results["Balanced_accuracy_mean"]
)

ablation_results["Macro_F1_drop_vs_full"] = (
    full_f1 - ablation_results["Macro_F1_mean"]
)

ablation_results = ablation_results.sort_values(
    "Balanced_accuracy_mean",
    ascending=False
).reset_index(drop=True)

display(ablation_results)

In [ ]:
dummy_uniform_ba = baseline_comparison.loc[
    baseline_comparison["Model"] == "Dummy uniform",
    "Balanced_accuracy"
].iloc[0]

dummy_most_ba = baseline_comparison.loc[
    baseline_comparison["Model"] == "Dummy most frequent",
    "Balanced_accuracy"
].iloc[0]

dummy_strat_ba = baseline_comparison.loc[
    baseline_comparison["Model"] == "Dummy stratified",
    "Balanced_accuracy"
].iloc[0]

maxstat_null_mean = max_permutation_results["Selected_by_score_Balanced_accuracy"].mean()

null_labels = [
    f"Selected model\n({best_model_name}, k = {best_k})",
    "Max-statistic null\n(mean)",
    "Dummy uniform",
    "Dummy most frequent",
    "Dummy stratified"
]

null_values = [
    best_balanced_accuracy,
    maxstat_null_mean,
    dummy_uniform_ba,
    dummy_most_ba,
    dummy_strat_ba
]

null_colors = ["#4C78A8", "#9E9E9E", "#C0C0C0", "#C0C0C0", "#C0C0C0"]

plot_ablation = ablation_results.copy()

ablation_labels = [
    f"{row.Feature_set}\n({int(row.Number_of_features):,})"
    for _, row in plot_ablation.iterrows()
]

ablation_values = plot_ablation["Balanced_accuracy_mean"].tolist()

ablation_colors = [
    "#C44E52" if fs == "Metadata only" else "#4C78A8"
    for fs in plot_ablation["Feature_set"]
]

chance_level = 1 / 3

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
plt.subplots_adjust(wspace=0.35)

# Left panel
ax = axes[0]
y_pos = np.arange(len(null_labels))

bars = ax.barh(
    y_pos,
    null_values,
    color=null_colors,
    edgecolor="white",
    height=0.7
)

ax.set_yticks(y_pos)
ax.set_yticklabels(null_labels, fontsize=10)
ax.invert_yaxis()

ax.set_xlim(0, max(1.05, best_balanced_accuracy + 0.25))
ax.set_xlabel("Balanced accuracy", fontsize=12)
ax.set_title("(a) Comparison with null references", fontsize=14, fontweight="bold", loc="left")

ax.axvline(chance_level, color="0.6", linestyle=":", linewidth=1.8)

for bar, val in zip(bars, null_values):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        ha="left",
        fontsize=10
    )

ax.text(
    0.98,
    0.88,
    f"max-stat\n$p$ = {selection_adjusted_p_value:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=11,
    color="#4C78A8"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Right panel
ax = axes[1]
y_pos = np.arange(len(ablation_labels))

bars = ax.barh(
    y_pos,
    ablation_values,
    color=ablation_colors,
    edgecolor="white",
    height=0.7
)

ax.set_yticks(y_pos)
ax.set_yticklabels(ablation_labels, fontsize=10)
ax.invert_yaxis()

ax.set_xlim(0, max(0.9, max(ablation_values) + 0.15))
ax.set_xlabel("Balanced accuracy", fontsize=12)
ax.set_title("(b) Feature-set ablation", fontsize=14, fontweight="bold", loc="left")

ax.axvline(chance_level, color="0.6", linestyle=":", linewidth=1.8)
ax.text(
    chance_level + 0.005,
    -0.45,
    "chance",
    color="0.5",
    fontsize=10
)

for bar, val in zip(bars, ablation_values):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        ha="left",
        fontsize=10
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

combined_png = figure_folder / "maxstat_null_reference_and_feature_ablation.png"
combined_pdf = figure_folder / "maxstat_null_reference_and_feature_ablation.pdf"

plt.savefig(combined_png, dpi=300, bbox_inches="tight")
plt.savefig(combined_pdf, bbox_inches="tight")

print("Saved:", combined_png)
print("Saved:", combined_pdf)

plt.show()

In [ ]:
def categorise_feature(feature):
    f = str(feature)
    fl = f.lower()
    
    if f.startswith("Morgan_"):
        return "Morgan substructure fingerprint"
    
    metadata_prefixes = [
        "Functional role_",
        "Chemical Family_",
        "Oral Dosage Form_",
        "Polymer Class_",
        "Polymer Flag_",
        "Viscosity_",
        "Aqueous Solubility_",
        "Solubility Class_"
    ]
    
    for prefix in metadata_prefixes:
        if f.startswith(prefix):
            return "Curated formulation metadata"
    
    if f == "Molecular Weight":
        return "Molecular size / mass"
    
    if any(token in fl for token in ["ecindex", "chi", "kier", "balaban", "zagreb", "topo", "diameter", "radius"]):
        return "Molecular topology / shape"
    
    if any(token in fl for token in ["logp", "slogp", "xlogp", "mrl", "apol"]):
        return "Hydrophobicity / polarizability"
    
    if any(token in fl for token in ["tpsa", "hbd", "hba", "nacid", "nbasic", "charge", "peoe", "estate"]):
        return "Polarity / charge / H-bonding"
    
    if any(token in fl for token in ["ring", "arom", "nrot", "rotatable"]):
        return "Rings / aromaticity / flexibility"
    
    if any(token in fl for token in ["aats", "atsc", "mats", "gats", "mor"]):
        return "Autocorrelation / electronic distribution"
    
    if any(token in fl for token in ["atom", "bond", "frag", "path", "walk"]):
        return "Atom / bond counts and fragments"
    
    return "Other Mordred descriptor"


def describe_feature(feature):
    f = str(feature)
    
    if f.startswith("Morgan_"):
        return "Circular fingerprint bit representing a local molecular substructure."
    
    if f.startswith("Functional role_"):
        return "Curated formulation role metadata."
    
    if f.startswith("Chemical Family_"):
        return "Curated chemical-family metadata."
    
    if f.startswith("Aqueous Solubility_") or f.startswith("Solubility Class_"):
        return "Curated solubility-related metadata."
    
    if f.startswith("Polymer"):
        return "Curated polymer-related metadata."
    
    if "ECIndex" in f:
        return "Topological descriptor related to molecular connectivity and branching."
    
    if any(x in f for x in ["GATS", "AATS", "MATS", "ATSC"]):
        return "Autocorrelation descriptor describing the distribution of atomic properties across the molecular graph."
    
    if "TopoPSA" in f or "TPSA" in f:
        return "Polar surface area descriptor related to polarity and hydrogen-bonding capacity."
    
    if "LogP" in f or "SLogP" in f or "XLogP" in f:
        return "Hydrophobicity-related descriptor."
    
    if "nRing" in f or "Ring" in f or "Arom" in f:
        return "Ring or aromaticity-related descriptor."
    
    if "Molecular Weight" in f:
        return "Molecular size or mass descriptor."
    
    return "Molecular descriptor contributing to model discrimination."


def get_selected_feature_names(fitted_pipeline):
    missing_names = np.array(fitted_pipeline.named_steps["missingness"].keep_columns_)
    variance_mask = fitted_pipeline.named_steps["variance"].get_support()
    after_variance_names = missing_names[variance_mask]
    
    select_step = fitted_pipeline.named_steps["select"]
    
    if select_step.k == "all":
        return after_variance_names
    
    select_mask = select_step.get_support()
    
    return after_variance_names[select_mask]


# Use selected RF if selected model is RF; otherwise use RF k=50 as interpretability model.
if best_model_name == "Random Forest":
    importance_k = best_k
else:
    importance_k = 50

rf_importance_model = make_model(
    clone(models["Random Forest"]),
    importance_k
)

rf_importance_model.fit(X_raw, y)

rf_selected_features = get_selected_feature_names(rf_importance_model)
rf_importances = rf_importance_model.named_steps["model"].feature_importances_

rf_feature_importance_df = pd.DataFrame({
    "Feature": rf_selected_features,
    "RF_importance": rf_importances
})

rf_feature_importance_df = rf_feature_importance_df.sort_values(
    "RF_importance",
    ascending=False
).reset_index(drop=True)

rf_feature_importance_df["Feature_category"] = rf_feature_importance_df["Feature"].apply(categorise_feature)
rf_feature_importance_df["Chemical_interpretation"] = rf_feature_importance_df["Feature"].apply(describe_feature)

importance_by_category = (
    rf_feature_importance_df
    .groupby("Feature_category")
    .agg(
        Total_importance=("RF_importance", "sum"),
        Mean_importance=("RF_importance", "mean"),
        Max_importance=("RF_importance", "max"),
        Number_of_features=("Feature", "count")
    )
    .reset_index()
    .sort_values("Total_importance", ascending=False)
    .reset_index(drop=True)
)

display(rf_feature_importance_df.head(30))
display(importance_by_category)

In [ ]:
top_rf_features = rf_feature_importance_df.head(25).copy()

fig, ax = plt.subplots(figsize=(9, 8))

ax.barh(
    top_rf_features["Feature"][::-1],
    top_rf_features["RF_importance"][::-1]
)

ax.set_xlabel("Random forest feature importance")
ax.set_title("Top feature importance results", fontsize=14, fontweight="bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

rf_imp_png = figure_folder / "RF_feature_importance_top25.png"
rf_imp_pdf = figure_folder / "RF_feature_importance_top25.pdf"

plt.savefig(rf_imp_png, dpi=300, bbox_inches="tight")
plt.savefig(rf_imp_pdf, bbox_inches="tight")

print("Saved:", rf_imp_png)
print("Saved:", rf_imp_pdf)

plt.show()


fig, ax = plt.subplots(figsize=(8, 6))

ax.barh(
    importance_by_category["Feature_category"][::-1],
    importance_by_category["Total_importance"][::-1]
)

ax.set_xlabel("Total random forest importance")
ax.set_title("Feature importance by chemical category", fontsize=14, fontweight="bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

category_png = figure_folder / "feature_importance_by_chemical_category.png"
category_pdf = figure_folder / "feature_importance_by_chemical_category.pdf"

plt.savefig(category_png, dpi=300, bbox_inches="tight")
plt.savefig(category_pdf, bbox_inches="tight")

print("Saved:", category_png)
print("Saved:", category_pdf)

plt.show()

In [ ]:
def morgan_bitvect(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )


ad_table = cv_prediction_df.copy().reset_index(drop=True)
ad_table["canonical_smiles"] = labelled_desc["canonical_smiles"].reset_index(drop=True).values

fps = [
    morgan_bitvect(s)
    for s in ad_table["canonical_smiles"]
]

similarity_rows = []

for i in range(len(ad_table)):
    for j in range(len(ad_table)):
        if i == j:
            continue
        
        if fps[i] is None or fps[j] is None:
            sim = np.nan
        else:
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
        
        similarity_rows.append({
            "Query_Index": i,
            "Neighbour_Index": j,
            "Query_Excipient": ad_table.loc[i, "Display_Name"],
            "Neighbour_Excipient": ad_table.loc[j, "Display_Name"],
            "Query_Label": ad_table.loc[i, "True_Label"],
            "Neighbour_Label": ad_table.loc[j, "True_Label"],
            "Same_Label": ad_table.loc[i, "True_Label"] == ad_table.loc[j, "True_Label"],
            "Tanimoto": sim
        })

similarity_df = pd.DataFrame(similarity_rows)

SIMILARITY_THRESHOLD = 0.65

ad_records = []

for i in range(len(ad_table)):
    sub = similarity_df[similarity_df["Query_Index"] == i].copy()
    sub = sub.dropna(subset=["Tanimoto"])
    
    same = sub[sub["Same_Label"] == True]
    different = sub[sub["Same_Label"] == False]
    
    max_same = same["Tanimoto"].max() if same.shape[0] > 0 else np.nan
    max_diff = different["Tanimoto"].max() if different.shape[0] > 0 else np.nan
    
    if sub.shape[0] > 0:
        nearest_row = sub.sort_values("Tanimoto", ascending=False).iloc[0]
        nearest_name = nearest_row["Neighbour_Excipient"]
        nearest_label = nearest_row["Neighbour_Label"]
        nearest_tanimoto = nearest_row["Tanimoto"]
    else:
        nearest_name = np.nan
        nearest_label = np.nan
        nearest_tanimoto = np.nan
    
    if pd.notna(max_same) and max_same >= SIMILARITY_THRESHOLD:
        group = "Same-label structural congener"
    elif pd.notna(max_diff) and max_diff >= SIMILARITY_THRESHOLD:
        group = "Different-label nearest congener"
    else:
        group = "No close structural congener"
    
    ad_records.append({
        "Display_Name": ad_table.loc[i, "Display_Name"],
        "True_Label": ad_table.loc[i, "True_Label"],
        "CV_Predicted_Label": ad_table.loc[i, "CV_Predicted_Label"],
        "Correct": ad_table.loc[i, "Correct"],
        "Max_same_label_Tanimoto": max_same,
        "Max_different_label_Tanimoto": max_diff,
        "Nearest_Neighbour": nearest_name,
        "Nearest_Neighbour_Label": nearest_label,
        "Nearest_Neighbour_Tanimoto": nearest_tanimoto,
        "Structural_Neighbourhood": group
    })

ad_df = pd.DataFrame(ad_records)

ad_summary_rows = []

for group, sub in ad_df.groupby("Structural_Neighbourhood"):
    correct = int(sub["Correct"].sum())
    total = int(sub.shape[0])
    prop = correct / total if total > 0 else np.nan
    ci_low, ci_high = wilson_score_interval(correct, total)
    
    ad_summary_rows.append({
        "Structural_Neighbourhood": group,
        "Correct": correct,
        "Total": total,
        "Proportion_correct": prop,
        "Wilson_CI_lower": ci_low,
        "Wilson_CI_upper": ci_high
    })

ad_summary = pd.DataFrame(ad_summary_rows)

display(ad_df)
display(ad_summary)

same_label_group = ad_df[
    ad_df["Structural_Neighbourhood"] == "Same-label structural congener"
]

no_congener_group = ad_df[
    ad_df["Structural_Neighbourhood"] == "No close structural congener"
]

if same_label_group.shape[0] > 0 and no_congener_group.shape[0] > 0:
    same_correct = int(same_label_group["Correct"].sum())
    same_total = int(same_label_group.shape[0])
    
    no_correct = int(no_congener_group["Correct"].sum())
    no_total = int(no_congener_group.shape[0])
    
    fisher_table = np.array([
        [same_correct, same_total - same_correct],
        [no_correct, no_total - no_correct]
    ])
    
    odds_ratio, fisher_p = fisher_exact(
        fisher_table,
        alternative="greater"
    )
else:
    odds_ratio = np.nan
    fisher_p = np.nan
    print("Fisher exact test not calculated because one group is empty.")

print("Applicability-domain Fisher exact p-value:", fisher_p)

In [ ]:
final_model = make_model(
    clone(models[best_model_name]),
    best_k
)

final_model.fit(X_raw, y)

pool_pred = final_model.predict(X_pool_raw)
pool_proba = final_model.predict_proba(X_pool_raw)

class_names = final_model.named_steps["model"].classes_

prediction_df = unlabelled_desc.copy()

prediction_df["Predicted_Label"] = pool_pred
prediction_df["Max_Probability"] = pool_proba.max(axis=1)
prediction_df["Uncertainty"] = 1 - prediction_df["Max_Probability"]

for i, cls in enumerate(class_names):
    prediction_df[f"Probability_{cls}"] = pool_proba[:, i]

prediction_df["Entropy"] = -np.sum(
    np.clip(pool_proba, 1e-12, 1) * np.log(np.clip(pool_proba, 1e-12, 1)),
    axis=1
)

prediction_df = prediction_df.sort_values(
    by=["Entropy", "Uncertainty"],
    ascending=False
).reset_index(drop=True)

prediction_df["Active_Learning_Rank"] = np.arange(1, prediction_df.shape[0] + 1)

top20_candidates = prediction_df.head(20).copy()
top50_candidates = prediction_df.head(50).copy()

print("Predicted unlabelled excipients:", prediction_df.shape[0])
print("Unique predicted canonical SMILES:", prediction_df["canonical_smiles"].nunique())

print("\nPredicted label distribution:")
print(prediction_df["Predicted_Label"].value_counts())

display(
    top20_candidates[
        [
            "Active_Learning_Rank",
            phase1_name_col,
            "Predicted_Label",
            "Max_Probability",
            "Uncertainty",
            "Entropy"
        ]
    ]
)

In [ ]:
summary_final = pd.DataFrame({
    "Metric": [
        "Seed",
        "Phase 1 file",
        "Label file",
        "Phase 1 records",
        "Input label rows",
        "Usable labels after cleaning",
        "Conflicting labels excluded",
        "Unmatched labels",
        "Valid labelled samples",
        "Promote labelled samples",
        "Neutral labelled samples",
        "Prevent labelled samples",
        "Unlabelled valid records predicted",
        "Unlabelled unique canonical SMILES",
        "Mordred descriptors",
        "Morgan fingerprint bits",
        "Metadata-derived features",
        "Total hybrid features",
        "Best model by composite score",
        "Best selected features",
        "Accuracy",
        "Balanced accuracy",
        "Macro F1",
        "Promote recall",
        "Selection score",
        "Out-of-fold accuracy",
        "Out-of-fold accuracy CI lower",
        "Out-of-fold accuracy CI upper",
        "Max-stat permutations completed",
        "Selection-adjusted max-statistic p-value",
        "Conservative max-BA p-value",
        "Mean selection-adjusted null BA",
        "Mean max-null BA",
        "Applicability-domain Fisher exact p-value"
    ],
    "Value": [
        SEED,
        phase1_path.name,
        label_path.name,
        phase1.shape[0],
        labels.shape[0],
        label_final.shape[0],
        label_conflicts.shape[0],
        unmatched.shape[0],
        labelled_for_ml.shape[0],
        y.value_counts().get("Promote", 0),
        y.value_counts().get("Neutral", 0),
        y.value_counts().get("Prevent", 0),
        prediction_df.shape[0],
        prediction_df["canonical_smiles"].nunique(),
        mordred_df.shape[1],
        morgan_df.shape[1],
        len(metadata_cols),
        len(feature_cols),
        best_model_name,
        best_k,
        best_accuracy,
        best_balanced_accuracy,
        best_macro_f1,
        best_promote_recall,
        best_selection_score,
        overall_accuracy,
        overall_ci[0],
        overall_ci[1],
        max_permutation_results.shape[0],
        selection_adjusted_p_value,
        max_ba_p_value,
        max_permutation_results["Selected_by_score_Balanced_accuracy"].mean(),
        max_permutation_results["Max_Balanced_accuracy"].mean(),
        fisher_p
    ]
})

excel_output = output_folder / "phase3_book16_maxstat_results.xlsx"
model_output = output_folder / "final_phase3_book16_maxstat_model.joblib"

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    summary_final.to_excel(writer, sheet_name="Summary", index=False)
    
    labels.to_excel(writer, sheet_name="Input_Labels", index=False)
    label_summary.to_excel(writer, sheet_name="Label_Summary", index=False)
    label_conflicts.to_excel(writer, sheet_name="Conflicting_Labels", index=False)
    matched_all.to_excel(writer, sheet_name="Matched_All_Rows", index=False)
    unmatched.to_excel(writer, sheet_name="Unmatched_Labels", index=False)
    labelled_invalid_smiles.to_excel(writer, sheet_name="Invalid_Labelled_SMILES", index=False)
    labelled_for_ml.to_excel(writer, sheet_name="Final_Training_Set", index=False)
    
    pca_df.to_excel(writer, sheet_name="PCA_Coordinates", index=False)
    pca_loadings.to_excel(writer, sheet_name="PCA_Loadings", index=False)
    
    model_comparison.to_excel(writer, sheet_name="Model_Comparison", index=False)
    classification_report_df.to_excel(writer, sheet_name="Classification_Report")
    confusion_matrix_df.to_excel(writer, sheet_name="Confusion_Matrix")
    cv_prediction_df.to_excel(writer, sheet_name="CV_Predictions", index=False)
    per_class_ci.to_excel(writer, sheet_name="Per_Class_Wilson_CI", index=False)
    
    baseline_comparison.to_excel(writer, sheet_name="Baseline_Comparison", index=False)
    
    max_stat_summary.to_excel(writer, sheet_name="Max_Statistic_Summary", index=False)
    max_permutation_results.to_excel(writer, sheet_name="Max_Statistic_Results", index=False)
    
    ablation_results.to_excel(writer, sheet_name="Ablation_Fixed_RF_k50", index=False)
    
    rf_feature_importance_df.to_excel(writer, sheet_name="RF_Feature_Importance", index=False)
    importance_by_category.to_excel(writer, sheet_name="Importance_By_Category", index=False)
    
    ad_df.to_excel(writer, sheet_name="Applicability_Domain", index=False)
    ad_summary.to_excel(writer, sheet_name="Applicability_Summary", index=False)
    similarity_df.to_excel(writer, sheet_name="Pairwise_Similarity", index=False)
    
    prediction_df.to_excel(writer, sheet_name="Pool_Predictions", index=False)
    top20_candidates.to_excel(writer, sheet_name="Top20_Active_Learning", index=False)
    top50_candidates.to_excel(writer, sheet_name="Top50_Active_Learning", index=False)

joblib.dump(final_model, model_output)

print("Saved Excel:")
print(excel_output)

print("\nSaved model:")
print(model_output)

print("\nSaved figures folder:")
print(figure_folder)

print("\nImportant final values:")
display(summary_final)